In [22]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from layers import MLP

### teacher model

In [23]:
class JointEncoder(nn.Module):
    def __init__(
        self,
        num_features,
        z_dim, # 각 feature의 출력층 dim
        enc_hidden_dim, # hidden layer의 dim
        enc_num_hidden # hidden layer 개수
    ):
        super().__init__()
        self.num_features = num_features

        self.feature_mlps = nn.ModuleList([
            MLP(
                in_dim=2, # [x*m, m]
                hidden_dim=enc_hidden_dim,
                out_dim=z_dim,
                num_hidden=enc_num_hidden
            )
            for _ in range(num_features)
        ])

    def forward(self, x, m):
        z_concat = []

        for idx in range(self.num_features):
            x_idx = x[:, idx]
            m_idx = m[:, idx]

            x_masked = x_idx * m_idx
            h = torch.stack([x_masked, m_idx], dim=-1)
            z = self.feature_mlps[idx](h)
            z_concat.append(z.unsqueeze(1))

        z = torch.cat(z_concat, dim=1)

        return z

In [24]:
class TeacherModel(nn.Module):
    def __init__(
        self,
        num_features,
        z_dim,
        enc_hidden_dim,
        enc_num_hidden,
        dec_hidden_dim, # 디코더 hidden layer의 dim
        dec_num_hidden, # 디코더 hidden layer 개수
        out_dim # 클래수 개수 (regression에서는 1)
    ):
        super().__init__()
        dec_input_dim = num_features * z_dim

        self.encoder = JointEncoder(
            num_features=num_features,
            z_dim=z_dim,
            enc_hidden_dim=enc_hidden_dim,
            enc_num_hidden=enc_num_hidden
        )

        self.predictor = MLP(
            in_dim=dec_input_dim,
            hidden_dim=dec_hidden_dim,
            out_dim=out_dim,
            num_hidden=dec_num_hidden
        )

    def forward(self, x, m):
        z = self.encoder(x, m)
        B, D, Z = z.shape # batch, feature dim, z dim
        z = z.view(B, D * Z)

        logit = self.predictor(z)
        return logit

In [25]:
# contrastive loss와의 확장을 위해 따로 미리 만들어둠
class TeacherLoss(nn.Module):
    def __init__(
            self, 
            teacher_model,
            task_type
        ):
        super().__init__()
        self.teacher = teacher_model
        self.task_type = task_type

        if task_type == "multi_classification":
            self.criterion = nn.CrossEntropyLoss()

        elif task_type == "binary_classification":
            self.criterion = nn.BCEWithLogitsLoss()

        elif task_type == "regression":
            self.criterion = nn.MSELoss()

    def forward(self, x, m, y):
        logit = self.teacher(x, m)

        if self.task_type == "multi_classification":
            loss = self.criterion(logit, y)

        elif self.task_type == "binary_classification":
            logit = logit.view(-1)
            y = y.view(-1).float()
            loss = self.criterion(logit, y)

        elif self.task_type == "regression":
            logit = logit.view(-1)
            y = y.view(-1).float()
            loss = self.criterion(logit, y)

        return {
            "loss": loss,
            "logit": logit,
        }

### student model

In [26]:
class StudentModel(nn.Module):
    def __init__(
        self,
        num_features,
        z_dim,
        enc_hidden_dim,
        enc_num_hidden,
        dec_hidden_dim, # 디코더 hidden layer의 dim
        dec_num_hidden, # 디코더 hidden layer 개수
        out_dim # 클래수 개수 (regression에서는 1)
    ):
        super().__init__()
        dec_input_dim = num_features * z_dim

        self.encoder = JointEncoder(
            num_features=num_features,
            z_dim=z_dim,
            enc_hidden_dim=enc_hidden_dim,
            enc_num_hidden=enc_num_hidden
        )

        self.predictor = MLP(
            in_dim=dec_input_dim,
            hidden_dim=dec_hidden_dim,
            out_dim=out_dim,
            num_hidden=dec_num_hidden
        )

    def forward(self, x, m):
        z = self.encoder(x, m)
        B, D, Z = z.shape
        z = z.view(B, D * Z)

        logit = self.predictor(z)
        return logit

In [27]:
class StudentLoss(nn.Module):
    def __init__(
      self,
      student_model,
      teacher_model,
      task_type,
      lambda_distill, # distill loss
      lambda_pred # prediction loss
    ):
        super().__init__()
        self.student = student_model
        self.teacher = teacher_model
        self.task_type = task_type
        self.lambda_distill = lambda_distill
        self.lambda_pred = lambda_pred
        self.mse = nn.MSELoss()

        if task_type == "multi_classification":
            self.criterion = nn.CrossEntropyLoss()

        elif task_type == "binary_classification":
            self.criterion = nn.BCEWithLogitsLoss()

        elif task_type == "regression":
            self.criterion = nn.MSELoss()

    def forward(self, x_full, x_masked, m_masked, y):
        '''
        x_full: teacher가 보는 full feature
        x_masked: student가 보는 masked feature
        m_masked: student mask (관측:1, 결측:0)
        '''
        # distill loss
        with torch.no_grad():
            m_full = torch.ones_like(x_full) # teacher는 mask가 모두 1
            z_teacher = self.teacher.encoder(x_full, m_full)
        
        z_student = self.student.encoder(x_masked, m_masked)
        logit_student = self.student(x_masked, m_masked)

        loss_distill = self.mse(z_student, z_teacher)

        # prediction loss
        if self.task_type == "multi_classification":
            loss_sup = self.criterion(logit_student, y)

        elif self.task_type == "binary_classification":
            logit_flat = logit_student.view(-1)
            y_flat = y.view(-1).float()
            loss_sup = self.criterion(logit_flat, y_flat)

        elif self.task_type == "regression":
            logit_flat = logit_student.view(-1)
            y_flat = y.view(-1).float()
            loss_sup = self.criterion(logit_flat, y_flat)

        # total loss
        loss = self.lambda_distill * loss_distill + self.lambda_pred * loss_sup

        return {
            "loss": loss,
            "loss_distill": loss_distill,
            "loss_sup": loss_sup,
            "logit_student": logit_student,
        }


In [ ]:
class JointFeatureAcquisition():
    def __init__(self, x, m, predictor, alpha=1, gamma=0):
        self.x = x
        self.m = m
        self.predictor = predictor
        self.alpha = alpha
        self.gamma = gamma

    def entropy(self, p,  eps=1e-10):
        alpha = self.alpha
        p = np.clip(p, eps, 1.0)  

        if alpha == 0.0:
            return np.log((p > eps).sum(axis=-1) + eps)
        
        elif alpha == 1.0:
            return -np.sum(p * np.log(p + eps), axis=-1)

        elif alpha > 1000:
            return -np.log(np.max(p, axis=-1) + eps)
        
        else:
            return (1.0 / (1.0 - alpha)) * np.log(np.sum(np.power(p, alpha), axis=-1) + eps)

    def alpha_gamma_cmi(self):
        x = self.x
        m = self.m
        gamma = self.gamma
        predictor = self.predictor

        device = next(predictor.parameters()).device

        m_upsampled = np.random.binomial(n=1, p=gamma, size=m.shape) # 각 feature별로 0 또는 1로 변형 
        m_repeated = np.maximum(m, m_upsampled) # 위에서는 모든 feature별로 진행했으니 max로 병합
        
        m_repeated = torch.tensor(m_repeated, dtype=torch.float32, device=device)

        with torch.no_grad():
            z_base = predictor.encoder(x, m_repeated) # 기본 z 얻어놓기
        B, D, Z = z_base.shape

        out = []

        # 변수 하나씩 mask해가며 entropy 계산
        for f in range(D):
            # without 계산
            m_without = m_repeated.clone()
            m_without[:, f] = 0.0
            z_without = z_base * m_without.unsqueeze(-1)

            with torch.no_grad():
                logits_without = predictor.predictor(z_without.view(B, D * Z))
                p_without = torch.softmax(logits_without, dim=-1).cpu().numpy()
            h_without = self.entropy(p=p_without)

            # with 계산
            m_with = m_repeated.clone()
            m_with[:, f] = 1.0
            z_with = z_base * m_with.unsqueeze(-1)

            with torch.no_grad():
                logits_with = predictor.predictor(z_with.view(B, D * Z))
                p_with = torch.softmax(logits_with, dim=-1).cpu().numpy()
            h_with = self.entropy(p=p_with)

            # 차이 계산
            entropy_diff = h_without - h_with
            out.append(entropy_diff)
            
        return np.stack(out, axis=-1)

    def acquire(self):
        m = self.m
        scores = self.alpha_gamma_cmi()
        scores -= scores.min()
        scores *= (1 - m)
        scores += 1e-10 * (1 - m) * np.random.uniform(size=(scores.shape)) # 최고점 score 점수 같음 방지

        selected = np.argmax(scores, axis=-1)
        m[np.arange(m.shape[0]), selected] = 1.0
        self.m = m # acquire 후 해당 feature의 mask = 1로 변경 

        return m, selected

### Cube

In [29]:
DATA_DIR = os.path.join("data", "cube")

X_train = torch.load(f"{DATA_DIR}/X_train_cdf.pt").float()
y_train = torch.load(f"{DATA_DIR}/y_train.pt").long()

X_val   = torch.load(f"{DATA_DIR}/X_val_cdf.pt").float()
y_val   = torch.load(f"{DATA_DIR}/y_val.pt").long()

X_test = torch.load(f"{DATA_DIR}/X_test_cdf.pt").float()
y_test = torch.load(f"{DATA_DIR}/y_test.pt").long()